In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load data - NO skiprows needed for your file
df = pd.read_csv('../data/kenya.csv') 

# Clean column names (removes any accidental spaces)
df.columns = [c.strip() for c in df.columns]

# Add Country column
df['Country'] = 'Kenya'

# Convert YEAR and DOY to a proper datetime column
# %Y%j is the format for Year + Day of Year
df['Date'] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract Month for seasonal analysis
df['Month'] = df['Date'].dt.month

print("Success! Step 1 Complete: Data loaded and dates parsed.")
df.head()

Success! Step 1 Complete: Data loaded and dates parsed.


,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,Month
0,2015,1,19.56,28.99,12.09,16.90,0.00,45.32,3.12,4.76,83.68,6.88,Kenya,2015-01-01,1
1,2015,2,19.63,29.77,11.04,18.73,0.00,38.76,3.23,4.35,83.67,5.85,Kenya,2015-01-02,1
2,2015,3,20.40,30.57,11.71,18.86,0.00,41.75,3.46,4.68,83.69,6.65,Kenya,2015-01-03,1
3,2015,4,21.33,31.20,13.02,18.18,3.49,51.87,2.29,4.00,83.62,8.60,Kenya,2015-01-04,1
4,2015,5,20.41,29.52,12.38,17.14,1.79,48.04,1.77,4.05,83.54,7.64,Kenya,2015-01-05,1


In [2]:
# 1. Replace NASA sentinel -999 with NaN (Not a Number)
df.replace(-999, np.nan, inplace=True)

# 2. Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")
df.drop_duplicates(inplace=True)

# 3. Missing Value Report
null_report = df.isna().sum()
null_pct = (null_report / len(df)) * 100
print("\nPercentage of missing values per column:")
print(null_pct[null_pct > 0])

# 4. Cleaning: Fill missing values using Forward Fill
# This takes the value from the previous day and fills the gap
df.ffill(inplace=True)

print("\nCleaning Complete!")

Duplicate rows found: 0

Percentage of missing values per column:
Series([], dtype: float64)

Cleaning Complete!


In [3]:
# Select numeric columns for outlier check
cols_to_check = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M']

# Calculate Z-scores
z_scores = np.abs(stats.zscore(df[cols_to_check]))

# Identify rows with outliers
outlier_rows = (z_scores > 3).any(axis=1)
print(f"Total rows with extreme outliers for Kenya: {outlier_rows.sum()}")

# Export cleaned data (KPI Requirement)
df.to_csv('../data/kenya_clean.csv', index=False)
print("Cleaned data exported to data/kenya_clean.csv")

Total rows with extreme outliers for Kenya: 116
Cleaned data exported to data/kenya_clean.csv


Detected 116 rows with extreme outliers (∣Z∣>3). I have decided to retain these rows because, in climate analysis, these points often represent significant extreme weather events which are critical for policy and adaptation planning. 


In [ ]:
# Aggregate daily data to monthly (ME = Month End)
monthly_df = df.resample('ME', on='Date').agg({'T2M':'mean', 'PRECTOTCORR':'sum'})

# 1. Monthly Temperature Line Chart
plt.figure(figsize=(14, 5))
plt.plot(monthly_df.index, monthly_df['T2M'], color='orange', marker='o', linestyle='-')
plt.title('Monthly Average Temperature Trends - Kenya')
plt.ylabel('Temp (°C)')
plt.grid(True, alpha=0.3)
plt.show()

# 2. Monthly Rainfall Bar Chart
plt.figure(figsize=(14, 5))
plt.bar(monthly_df.index, monthly_df['PRECTOTCORR'], color='teal', alpha=0.7)
plt.title('Monthly Total Precipitation (Rainfall) - Kenya')
plt.ylabel('Total Rainfall (mm)')
plt.show()

In [ ]:
# 1. Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[cols_to_check].corr(), annot=True, cmap='YlGnBu', fmt=".2f")
plt.title('Correlation Heatmap - Kenya')
plt.show()

# 2. Bubble Chart: Temp vs Humidity (Size = Rainfall)
sample_df = df.sample(500, random_state=42)
plt.figure(figsize=(10, 7))
scatter = plt.scatter(sample_df['T2M'], sample_df['RH2M'], 
                      s=sample_df['PRECTOTCORR']*15, alpha=0.5, 
                      c=sample_df['PRECTOTCORR'], cmap='viridis')
plt.title('Kenya: Temp vs Humidity (Bubble Size = Rainfall)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Humidity (%)')
plt.colorbar(scatter, label='Precipitation (mm)')
plt.show()

# Kenya Climate Analysis: Negotiation-Grade Insights

### 1. What is changing? (Trend)
Kenya's climate displays a stable seasonal temperature trend, but high precipitation volatility. While most days are relatively dry, the dataset captures extreme rainfall events (as seen in the large bubbles) that occur primarily in the 18-21°C range.

### 2. What did it cause? (Impact)
The correlation heatmap confirms a **strong negative correlation (-0.79)** between Maximum Temperature and Humidity. This indicates that as heat increases, the air dries out rapidly, which can lead to crop wilting. Conversely, the bubble chart shows that extreme rainfall is "locked" to high-humidity windows, making flash floods highly predictable if humidity thresholds are monitored.

### 3. What does it demand? (Policy)
To protect Kenya’s agricultural sector ahead of COP32, we demand **investment in 'Buffer Infrastructure'** (such as rainwater harvesting and soil moisture sensors). Since extreme rain is concentrated in specific temperature windows, farmers can be alerted to prepare for planting or flood-proofing based on real-time humidity monitoring.